# HomecareCCV - ML con outcomes reales por cohorte\n\nEste notebook reproduce el pipeline corregido de HomecareCCV. La versión anterior entrenaba contra `risk_level`, una etiqueta sintética generada por una regla determinista, por lo que las métricas cercanas a 0.99 eran un artefacto.\n\nAquí entrenamos y evaluamos modelos contra desenlaces reales separados por cohorte: `stroke`, `cardio` y `HeartDisease`. La regla MEWS/Framingham se conserva como baseline clínico auditable, no como target de ML.

## 1. Preparar entorno Colab\n\nEjecuta esta celda en Colab. Instala Kaggle y clona la rama con el pipeline corregido.

In [ ]:
!pip -q install kaggle\n!rm -rf homecare\n!git clone -b feat/real-outcome-ml https://github.com/cmorregof/homecare.git homecare\n%cd homecare\n!pip -q install -r backend/requirements.txt

## 2. Subir `kaggle.json` y descargar datasets reales\n\nÚnico paso manual: descarga tu API token desde Kaggle (`Account > Create New Token`) y sube el archivo `kaggle.json` cuando Colab lo pida. No se usan datos sintéticos ni smoke tests.

In [ ]:
from google.colab import files\nfrom pathlib import Path\n\nuploaded = files.upload()\nPath('/root/.kaggle').mkdir(parents=True, exist_ok=True)\n!cp kaggle.json /root/.kaggle/kaggle.json\n!chmod 600 /root/.kaggle/kaggle.json\n\n!mkdir -p data/mock\n!kaggle datasets download fedesoriano/stroke-prediction-dataset -p data/mock/ --unzip\n!kaggle datasets download sulianova/cardiovascular-disease-dataset -p data/mock/ --unzip\n!kaggle datasets download fedesoriano/heart-failure-prediction -p data/mock/ --unzip\n!ls -lh data/mock

## 3. Ejecutar pipeline real-outcome\n\nEl script hace lo siguiente: carga las tres cohortes, conserva el outcome real, elimina leakage, elimina columnas constantes/casi constantes, entrena modelos, calibra probabilidades, calcula curva de decisión, subgrupos, bootstrap y SHAP.

In [ ]:
!PYTHONPATH=backend python -m ml.real_outcomes --bootstrap-iterations 200

## 4. Cargar resultados y resumen principal

In [ ]:
import json\nimport pandas as pd\nfrom pathlib import Path\n\nresults_path = Path('backend/ml/models/real_outcomes/real_outcome_results.json')\nresults = json.loads(results_path.read_text())\nsummary = pd.DataFrame(results['summary'])\nsummary[[\n    'cohort', 'rows', 'outcome_prevalence', 'best_model',\n    'test_roc_auc', 'test_auc_pr', 'test_brier',\n    'rule_roc_auc', 'rule_auc_pr', 'rule_brier',\n    'delta_mean_net_benefit_vs_rule', 'leakage_passed'\n]]

## 5. Auditoría de leakage\n\nLa condición crítica es que el outcome real no aparezca convertido en feature. Si `leakage_passed` es `False`, el experimento es inválido.

In [ ]:
for cohort, payload in results['cohorts'].items():\n    print('\nCOHORTE:', cohort)\n    print('Outcome fuente:', payload['source_outcome'])\n    print('Leakage audit:', payload['leakage_audit'])\n    print('Features constantes/casi constantes removidas:', payload['constant_feature_audit']['dropped_features'])\n    assert payload['leakage_audit']['forbidden_features_absent_from_model_matrix'] is True\nprint('\nAuditoría anti-leakage aprobada.')

## 6. Comparativo ML vs score-regla\n\nEl score-regla MEWS/Framingham se evalúa contra el outcome real. No se usa como etiqueta de entrenamiento.

In [ ]:
comparison = summary.assign(\n    delta_roc_auc = summary['test_roc_auc'] - summary['rule_roc_auc'],\n    delta_auc_pr = summary['test_auc_pr'] - summary['rule_auc_pr'],\n    delta_brier = summary['test_brier'] - summary['rule_brier'],\n)[[\n    'cohort', 'best_model', 'test_roc_auc', 'rule_roc_auc', 'delta_roc_auc',\n    'test_auc_pr', 'rule_auc_pr', 'delta_auc_pr',\n    'test_brier', 'rule_brier', 'delta_brier',\n    'delta_mean_net_benefit_vs_rule'\n]]\ncomparison

## 7. Curvas de calibración y decisión\n\nEl pipeline guarda figuras para cada cohorte/modelo. Aquí mostramos las figuras del mejor modelo por cohorte.

In [ ]:
from IPython.display import display, Image\n\nfor cohort, payload in results['cohorts'].items():\n    best = payload['best_model']\n    figures = payload['models'][best]['figures']\n    print(f'\n{cohort} - {best} - calibración')\n    display(Image(filename=figures['calibration']))\n    print(f'{cohort} - {best} - curva de decisión')\n    display(Image(filename=figures['decision_curve']))

## 8. Subgrupos y SHAP\n\nEl JSON incluye desempeño por sexo/franja de edad y top factores SHAP del caso de mayor probabilidad en test.

In [ ]:
for cohort, payload in results['cohorts'].items():\n    best = payload['best_model']\n    model_payload = payload['models'][best]\n    print('\nCOHORTE:', cohort, '| MODELO:', best)\n    print('Subgrupos:')\n    display(pd.DataFrame(model_payload['subgroups']['sex']).T)\n    display(pd.DataFrame(model_payload['subgroups']['age_band']).T)\n    print('Top factores SHAP/proxy:')\n    display(pd.DataFrame(model_payload['shap_top_features']))

## 9. Mensaje metodológico clave\n\nLas métricas bajan frente al pipeline anterior porque ahora se evalúan desenlaces reales y se eliminó la fuga de información. Eso no es un retroceso: es una mejora de validez. Un AUC realista de 0.77-0.91, acompañado de calibración, curvas de decisión y auditoría de leakage, es más defendible científicamente que un 0.99 producido por una etiqueta determinística.